In [ ]:
# Section 1: Imports and Environment Setup
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import cv2
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set random seed for reproducibility
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Hyperparameters
batch_size = 8
num_epochs = 100
learning_rate = 1e-4
pretrain_epochs = 10

In [ ]:
# Section 2: Dataset Class
class ColorizationDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = cv2.imread(img_path)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        
        # Normalize L [0,1], ab [-1,1]
        l_channel = img_lab[:,:,0] / 255.0
        ab_channels = (img_lab[:,:,1:] - 128) / 128.0
        
        # Resize to 256x256
        l_channel = cv2.resize(l_channel, (256, 256), interpolation=cv2.INTER_AREA)
        ab_channels = cv2.resize(ab_channels, (256, 256), interpolation=cv2.INTER_AREA)
        
        l_channel = torch.from_numpy(l_channel).float().unsqueeze(0)
        ab_channels = torch.from_numpy(ab_channels.transpose((2, 0, 1))).float()
        
        if self.transform:
            l_channel = self.transform(l_channel)
            ab_channels = self.transform(ab_channels)
        
        return {'L': l_channel, 'ab': ab_channels}

# Download Dataset (Example: ImageNet subset or custom)
# Option 1: ImageNet subset (via torchvision, requires registration)
# from torchvision.datasets import ImageNet
# dataset = ImageNet('/path/to/imagenet', split='train', transform=transforms.ToTensor())

# Option 2: Custom Dataset (Manual Download)
# Download from: Wikimedia Commons, Flickr CC, or Kaggle (e.g., COCO)
# Example: Place images in 'data/heritage_images/'
image_dir = 'data/heritage_images'
image_paths = [os.path.join(image_dir, img) for img in os.listdir(image_dir) if img.endswith(('.jpg', '.png'))]
dataset = ColorizationDataset(image_paths, transform=transforms.RandomHorizontalFlip())

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Section 3: Model Definition
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 2, 4, 2, 1), nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 0), nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Initialize models
G = Generator().to(device)
D = Discriminator().to(device)

In [ ]:
# Section 4: Loss Functions and Optimizers
vgg = models.vgg16(pretrained=True).features.eval().to(device)
for param in vgg.parameters():
    param.requires_grad = False

def perceptual_loss(pred, target):
    pred_features = vgg(pred)
    target_features = vgg(target)
    return nn.functional.mse_loss(pred_features, target_features)

criterion_GAN = nn.BCELoss().to(device)
criterion_L1 = nn.L1Loss().to(device)

optimizer_G = optim.Adam(G.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=learning_rate, betas=(0.5, 0.999))

In [ ]:
# Section 5: Training Loop
def train_step(G, D, optimizer_G, optimizer_D, real_ab, L):
    batch_size = L.size(0)
    real_label = torch.ones(batch_size, 1, 30, 30).to(device)
    fake_label = torch.zeros(batch_size, 1, 30, 30).to(device)

    # Generator forward
    G.train()
    fake_ab = G(L)
    fake_img = torch.cat([L, fake_ab], dim=1)

    # Discriminator forward
    D.train()
    real_pred = D(torch.cat([L, real_ab], dim=1))
    fake_pred = D(fake_img.detach())

    # D loss
    d_loss_real = criterion_GAN(real_pred, real_label)
    d_loss_fake = criterion_GAN(fake_pred, fake_label)
    d_loss = (d_loss_real + d_loss_fake) / 2
    optimizer_D.zero_grad()
    d_loss.backward()
    optimizer_D.step()

    # G loss
    fake_pred = D(fake_img)
    g_gan_loss = criterion_GAN(fake_pred, real_label)
    g_l1_loss = criterion_L1(fake_ab, real_ab) * 100  # Lambda = 100
    g_perceptual_loss = perceptual_loss(fake_img, torch.cat([L, real_ab], dim=1))
    g_loss = g_gan_loss + g_l1_loss + g_perceptual_loss
    optimizer_G.zero_grad()
    g_loss.backward()
    optimizer_G.step()

    return g_loss.item(), d_loss.item()

# Pretraining Generator
print("Pretraining Generator...")
for epoch in range(pretrain_epochs):
    total_g_loss = 0
    for batch in tqdm(train_loader, desc=f"Pretrain Epoch {epoch+1}/{pretrain_epochs}"):
        L = batch['L'].to(device)
        real_ab = batch['ab'].to(device)
        fake_ab = G(L)
        g_l1_loss = criterion_L1(fake_ab, real_ab) * 100
        g_perceptual_loss = perceptual_loss(torch.cat([L, fake_ab], dim=1), torch.cat([L, real_ab], dim=1))
        g_loss = g_l1_loss + g_perceptual_loss
        optimizer_G.zero_grad()
        g_loss.backward()
        optimizer_G.step()
        total_g_loss += g_loss.item()
    print(f"Pretrain Epoch [{epoch+1}/{pretrain_epochs}], G Loss: {total_g_loss/len(train_loader):.4f}")

# GAN Training
print("Starting GAN Training...")
for epoch in range(num_epochs):
    total_g_loss, total_d_loss = 0, 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        L = batch['L'].to(device)
        real_ab = batch['ab'].to(device)
        g_loss, d_loss = train_step(G, D, optimizer_G, optimizer_D, real_ab, L)
        total_g_loss += g_loss
        total_d_loss += d_loss
    print(f"Epoch [{epoch+1}/{num_epochs}], G Loss: {total_g_loss/len(train_loader):.4f}, D Loss: {total_d_loss/len(train_loader):.4f}")

# Save model
torch.save(G.state_dict(), 'generator_final.pth')
torch.save(D.state_dict(), 'discriminator_final.pth')

In [ ]:
# Section 6: Inference
def colorize_image(image_path, G):
    G.eval()
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    l_channel = img_lab[:,:,0] / 255.0
    l_channel = cv2.resize(l_channel, (256, 256), interpolation=cv2.INTER_AREA)
    l_tensor = torch.from_numpy(l_channel).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        ab_pred = G(l_tensor)
    ab_pred = ab_pred.cpu().numpy().squeeze() * 128 + 128  # Denormalize ab
    l_channel = cv2.resize(l_channel, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_AREA) * 255
    ab_pred_resized = cv2.resize(ab_pred.transpose(1, 2, 0), (img.shape[1], img.shape[0]), interpolation=cv2.INTER_AREA)
    colorized = cv2.cvtColor(np.dstack((l_channel, ab_pred_resized)), cv2.COLOR_LAB2RGB)
    return colorized

# Visualize
sample_path = 'data/heritage_images/sample.jpg'
colorized_img = colorize_image(sample_path, G)
plt.imshow(colorized_img)
plt.title("Colorized Image")
plt.axis('off')
plt.show()